In [1]:
import pandas as pd
import numpy as np

# Step 1 - Load IMDb dataset
df_imdb = pd.read_csv('../data/Top_10000_Movies_IMDb.csv')

# Step 2 - Extract imdb_id from Link column (before dropping)
df_imdb['imdb_id'] = df_imdb['Link'].str.extract(r'(tt\d+)')

# Step 3 - Drop unnecessary columns
df_imdb.drop(['ID', 'Plot', 'Link', 'Stars', 'Directors'], axis=1, inplace=True)

# Step 4 - Extract primary genre (must be done before Metascore imputation)
df_imdb['Genre'] = df_imdb['Genre'].str.split(',').str[0].str.strip()

# Step 5 - Metascore missing value imputation (Genre-wise mean → overall mean)
df_imdb['Metascore'] = df_imdb.groupby('Genre')['Metascore']\
    .transform(lambda x: x.fillna(x.mean()))
df_imdb['Metascore'].fillna(df_imdb['Metascore'].mean(), inplace=True)

# Step 6 - Create Hit target variable (before capping)
# Hit = IMDb Rating >= 7.5 AND Votes >= 50,000 (80th percentile)
df_imdb['Hit'] = ((df_imdb['Rating'] >= 7.5) & (df_imdb['Votes'] >= 50000)).astype(int)

# Step 7 - Outlier treatment: Gross only (3*IQR Capping)
# Votes is not capped — high vote count is a meaningful signal for hit prediction
Q1 = df_imdb['Gross'].quantile(0.25)
Q3 = df_imdb['Gross'].quantile(0.75)
IQR = Q3 - Q1
df_imdb['Gross'] = df_imdb['Gross'].clip(
    lower=Q1 - 3*IQR,
    upper=Q3 + 3*IQR
)

# Step 8 - Remove decimal points from Gross
df_imdb['Gross'] = df_imdb['Gross'].astype(int)

# Step 9 - Verify
print('=== Missing Values ===')
print(df_imdb.isnull().sum())
print()
print('=== Shape ===')
print(df_imdb.shape)
print()
print('=== Hit Distribution ===')
print(df_imdb['Hit'].value_counts())
print(f'Hit ratio: {df_imdb["Hit"].mean()*100:.1f}%')
print()
print('=== Basic Statistics ===')
print(df_imdb.describe())

# Step 10 - Save
df_imdb.to_csv('../data/cleaned_imdb.csv', index=False)
print('\ncleaned_imdb.csv saved!')


=== Missing Values ===
Movie Name    0
Rating        0
Runtime       0
Genre         0
Metascore     0
Votes         0
Gross         0
imdb_id       0
Hit           0
dtype: int64

=== Shape ===
(9999, 9)

=== Hit Distribution ===
Hit
0    8953
1    1046
Name: count, dtype: int64
Hit ratio: 10.5%

=== Basic Statistics ===
            Rating    Metascore         Votes         Gross          Hit
count  9999.000000  9999.000000  9.999000e+03  9.999000e+03  9999.000000
mean      6.698620    58.667969  9.127103e+04  2.382198e+07     0.104610
std       0.852755    15.798014  1.684508e+05  3.678157e+07     0.306066
min       4.600000     7.000000  1.000100e+04  7.000000e+00     0.000000
25%       6.100000    49.000000  1.678500e+04  3.654100e+04     0.000000
50%       6.700000    58.000000  3.391200e+04  4.244155e+06     0.000000
75%       7.300000    68.000000  8.965600e+04  3.306295e+07     0.000000
max       9.300000   100.000000  2.752419e+06  1.321422e+08     1.000000

cleaned_imdb.csv s

In [2]:
# Step 1 - Load TMDB dataset
df_tmdb = pd.read_csv('../data/tmdb_movies_data.csv')

# Step 2 - Drop unnecessary columns
# Dropped: identifier columns, text data, columns duplicated with IMDb,
#          release_date (40%+ missing after merge), vote_average (corr=0.87 with Rating)
drop_cols = [
    'id', 'original_title', 'homepage', 'tagline',
    'keywords', 'overview', 'cast', 'director',
    'genres', 'runtime', 'release_date',
    'vote_count', 'budget', 'revenue', 'vote_average'
]
df_tmdb.drop(drop_cols, axis=1, inplace=True)

# Step 3 - Drop rows with missing imdb_id (merge key)
df_tmdb.dropna(subset=['imdb_id'], inplace=True)

# Step 4 - Fill missing production_companies with Unknown
df_tmdb['production_companies'].fillna('Unknown', inplace=True)

# Step 5 - Extract primary company for group imputation
df_tmdb['primary_company'] = df_tmdb['production_companies'].str.split('|').str[0].str.strip()

# Step 6 - budget_adj, revenue_adj missing value imputation
# 0 values represent missing data — replaced with NaN first
# 1st: fill with company-wise median / 2nd: fill remaining with overall median
for col in ['budget_adj', 'revenue_adj']:
    df_tmdb[col].replace(0, np.nan, inplace=True)
    df_tmdb[col] = df_tmdb.groupby('primary_company')[col]\
        .transform(lambda x: x.fillna(x.median()))
    overall_median = df_tmdb[col].median()
    df_tmdb[col].fillna(overall_median, inplace=True)

# Step 7 - Drop temporary primary_company column
df_tmdb.drop('primary_company', axis=1, inplace=True)

# Step 8 - Outlier treatment: IQR Capping for budget_adj, revenue_adj
for col in ['budget_adj', 'revenue_adj']:
    Q1 = df_tmdb[col].quantile(0.25)
    Q3 = df_tmdb[col].quantile(0.75)
    IQR = Q3 - Q1
    df_tmdb[col] = df_tmdb[col].clip(
        lower=Q1 - 1.5*IQR,
        upper=Q3 + 1.5*IQR
    )

# Step 9 - Verify
print('=== Missing Values ===')
print(df_tmdb.isnull().sum())
print()
print('=== Shape ===')
print(df_tmdb.shape)
print()
print('=== Basic Statistics ===')
print(df_tmdb.describe())

# Step 10 - Encode production_companies as binary is_major_company feature
# Major studios defined as widely recognized large-scale production companies
major = [
    'Universal Pictures', 'Warner Bros.', 'Paramount Pictures',
    'Columbia Pictures', 'Walt Disney Pictures',
    'Twentieth Century Fox Film Corporation', '20th Century Fox', 'Twentieth Century Fox',
    'Marvel Studios', 'Marvel Entertainment', 'DC Entertainment',
    'Pixar Animation Studios', 'DreamWorks Pictures', 'DreamWorks Animation',
    'Sony Pictures Entertainment', 'Metro-Goldwyn-Mayer (MGM)',
    'Lionsgate', 'Amblin Entertainment', 'Legendary Pictures',
    'Lucasfilm', 'Lucasfilm Ltd.', 'New Line Cinema',
    'Miramax Films', 'Touchstone Pictures',
]
df_tmdb['is_major_company'] = df_tmdb['production_companies']\
    .str.contains('|'.join(major)).astype(int)
df_tmdb.drop('production_companies', axis=1, inplace=True)

# Step 11 - Save
df_tmdb.to_csv('../data/tmdb_cleaned.csv', index=False)
print('tmdb_cleaned.csv saved!')


/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keep

=== Missing Values ===
imdb_id                 0
popularity              0
production_companies    0
release_year            0
budget_adj              0
revenue_adj             0
dtype: int64

=== Shape ===
(10856, 6)

=== Basic Statistics ===
         popularity  release_year    budget_adj   revenue_adj
count  10856.000000  10856.000000  1.085600e+04  1.085600e+04
mean       0.646828   2001.313928  2.539912e+07  5.022235e+07
std        1.000545     12.815353  2.244138e+07  5.255880e+07
min        0.000065   1960.000000  9.210911e-01  2.370705e+00
25%        0.207759   1995.000000  7.359997e+06  9.833310e+06
50%        0.384018   2006.000000  1.939207e+07  3.155521e+07
75%        0.714339   2011.000000  3.592018e+07  7.210981e+07
max       32.985763   2015.000000  7.876047e+07  1.655246e+08
tmdb_cleaned.csv saved!


/var/folders/z1/bmx7_sfx75b6yb0t8dzr1s8w0000gn/T/ipykernel_46271/607740234.py:70: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_tmdb['is_major_company'] = df_tmdb['production_companies']\


In [3]:
# Step 1 - Load cleaned datasets
df_imdb = pd.read_csv('../data/cleaned_imdb.csv')
df_tmdb = pd.read_csv('../data/tmdb_cleaned.csv')

print('IMDb shape:', df_imdb.shape)
print('TMDB shape:', df_tmdb.shape)

# Step 2 - Inner join on imdb_id
# Inner join selected over left join to ensure all features are complete (no NaN after merge)
df = pd.merge(df_imdb, df_tmdb, on='imdb_id', how='inner')

# Step 3 - Remove duplicate rows
# Tekken appears twice in TMDB — drop duplicates
df.drop_duplicates(inplace=True)

# Step 4 - Verify
print()
print('=== After Merge ===')
print('Shape:', df.shape)
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Columns ===')
print(df.columns.tolist())
print()
print('=== Hit Distribution ===')
print(df['Hit'].value_counts())
print(f'Hit ratio: {df["Hit"].mean()*100:.1f}%')
print()
print('=== Sample ===')
print(df.head())

# Step 5 - Save intermediate result
df.to_csv('../data/merged_movie_dataset.csv', index=False)
print('\nmerged_movie_dataset.csv (intermediate) saved!')


IMDb shape: (9999, 9)
TMDB shape: (10856, 6)

=== After Merge ===
Shape: (5979, 14)

=== Missing Values ===
Movie Name          0
Rating              0
Runtime             0
Genre               0
Metascore           0
Votes               0
Gross               0
imdb_id             0
Hit                 0
popularity          0
release_year        0
budget_adj          0
revenue_adj         0
is_major_company    0
dtype: int64

=== Columns ===
['Movie Name', 'Rating', 'Runtime', 'Genre', 'Metascore', 'Votes', 'Gross', 'imdb_id', 'Hit', 'popularity', 'release_year', 'budget_adj', 'revenue_adj', 'is_major_company']

=== Hit Distribution ===
Hit
0    5363
1     616
Name: count, dtype: int64
Hit ratio: 10.3%

=== Sample ===
                 Movie Name  Rating  Runtime      Genre  Metascore    Votes  \
0  The Shawshank Redemption     9.3  142 min      Drama       82.0  2752419   
1             The Godfather     9.2  175 min      Crime      100.0  1914751   
2           The Dark Knight     9.0

In [4]:
# Step 1 - Load merged dataset
df = pd.read_csv('../data/merged_movie_dataset.csv')

# Step 2 - Convert Runtime from string to integer
df['Runtime'] = df['Runtime'].str.replace(' min', '').astype(int)

# Step 3 - Log transformation on Votes (highly skewed distribution)
df['Votes'] = np.log1p(df['Votes'])

# Step 4 - One-hot encoding for Genre
# drop_first=True to avoid multicollinearity
df = pd.get_dummies(df, columns=['Genre'], drop_first=True)

# Step 5 - Convert bool columns to int for model compatibility
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

# Step 6 - Save final preprocessed dataset
df.to_csv('../data/merged_movie_dataset.csv', index=False)
print('merged_movie_dataset.csv (final) saved!')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())


merged_movie_dataset.csv (final) saved!
Shape: (5979, 29)
Columns: ['Movie Name', 'Rating', 'Runtime', 'Metascore', 'Votes', 'Gross', 'imdb_id', 'Hit', 'popularity', 'release_year', 'budget_adj', 'revenue_adj', 'is_major_company', 'Genre_Adventure', 'Genre_Animation', 'Genre_Biography', 'Genre_Comedy', 'Genre_Crime', 'Genre_Drama', 'Genre_Family', 'Genre_Fantasy', 'Genre_Horror', 'Genre_Music', 'Genre_Mystery', 'Genre_Romance', 'Genre_Sci-Fi', 'Genre_Thriller', 'Genre_War', 'Genre_Western']
